

1.Convert PDFs to images if needed:

* Use pdf2image or PyMuPDF to convert each page to an image.
* Save images in a subfolder like data/processed/images/.



In [ ]:
#!pip install pdf2image

In [ ]:
from pdf2image import convert_from_path
import os

def pdf_to_images(pdf_path, output_folder):
    pages = convert_from_path(pdf_path)
    os.makedirs(output_folder, exist_ok=True)
    for i, page in enumerate(pages):
        page_path = os.path.join(output_folder, f"page_{i+1}.png")
        page.save(page_path, "PNG")
    return [os.path.join(output_folder, f"page_{i+1}.png") for i in range(len(pages))]


Command to install paddlepaddle-gpu compatible with GoogleCollab
Change Runtime Environment of Collab to GPU then install.
!python -m pip install paddlepaddle-gpu==3.2.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/

In [ ]:
#!python -m pip install paddlepaddle-gpu==3.2.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/

Now install paddleocr


In [ ]:
#!pip install paddleocr

In [ ]:
# Install a compatible version of PyTorch
#!pip install torch==2.3.0 torchvision==0.18.0 torchaudio==2.3.0 --index-url https://download.pytorch.org/whl/cu118

In [ ]:
#!pip install paddle

In [ ]:
import paddle

print("Compiled with CUDA:", paddle.device.is_compiled_with_cuda())
print("CUDA device count:", paddle.device.cuda.device_count())
print("Current device:", paddle.device.get_device())

In [ ]:
print(paddle.set_device("gpu"))
print(paddle.get_device())

In [ ]:
from paddleocr import PaddleOCR

#---- text detection + text recognition using server model ----

# ocr = PaddleOCR(
#     use_doc_orientation_classify=False,
#     use_doc_unwarping=False,
#     use_textline_orientation=False)

#---- text image preprocessing + text detection + textline orientation classification + text recognition ------

# ocr = PaddleOCR(use_doc_orientation_classify=True, use_doc_unwarping=True)

#---- text image preprocessing + text detection + using mobile model ------
ocr = PaddleOCR(
    text_detection_model_name="PP-OCRv5_mobile_det",
    text_recognition_model_name="PP-OCRv5_mobile_rec",
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False) # Switch to PP-OCRv5_mobile models
result = ocr.predict("data/sample_image1.png")
for res in result:
    res.print()
    res.save_to_img("output")
    res.save_to_json("output")


🔹 Step 1. Download FUNSD

From HuggingFace:

In [ ]:
from datasets import load_dataset

dataset = load_dataset("nielsr/funsd")
print(dataset)
# → {'train': Dataset(...), 'test': Dataset(...)}

🔹 Step 2. Get Ground Truth from FUNSD

FUNSD provides JSON annotations per image. Each has text + bounding boxes.


In [ ]:
def extract_ground_truth(annotations):
    ground_truth = []
    for form_field in annotations["form"]:
        for word in form_field["words"]:
            ground_truth.append({
                "text": word["text"],
                "bbox": word["box"]
            })
    return ground_truth

### Layout test using PP-DocLayout_plus-L

In [ ]:
import time 
import os 
from pathlib import Path
from paddleocr import LayoutDetection
# Add the src directory to the Python path
OUTPUT = "../outputs"

layout_model = LayoutDetection(model_name="PP-DocLayout_plus-L")


image_path = Path("data/page_1.png")
output_folder = Path(OUTPUT) / "layout"
os.makedirs(output_folder, exist_ok=True)
before = time.time()
if image_path.exists():
    output = layout_model.predict(str(image_path))  
    after = time.time()
    for res in output:
        res.print()
        res.save_to_img(Path(OUTPUT) / "layout")
        res.save_to_json(Path(OUTPUT) / "layout")
    print(f"Processed {image_path} in {after - before:.2f} seconds")
else:
    print(f"Error: The file {image_path} does not exist.")
